<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/7_Elasticsearch_BM25_Compras_Claras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"></a>

# S07 — Elasticsearch Search Lab

**Tres recursos de la sesión:** presentación principal, este cuaderno y laboratorio activo.

La presentación explica los conceptos y los ejemplos. Este cuaderno ejecuta, automatiza y deja evidencia. El laboratorio organiza el ritmo de clase.

## Resultado esperado

Al terminar debes poder decir: sé crear un índice, definir mapping, usar Console, conectar Python, cargar documentos con `bulk`, buscar con Query DSL, interpretar `hits`, evaluar con Precision@5 y repetirlo con otro corpus.

## 0. Cómo se usa este cuaderno

No leas este archivo como un libro lineal. Lo ejecutas cuando la presentación lo indique.

La secuencia de aprendizaje es:

1. **Primero en Elasticsearch Console:** entiendes la API sin Python.
2. **Después en Python:** automatizas la misma operación.
3. **Finalmente en transferencia:** aplicas el mismo razonamiento a noticias.

Cada bloque importante tiene: objetivo, código, explicación, evidencia esperada y error frecuente.

# A · Console antes de Python

Antes de usar `client.search()`, ejecuta mentalmente o directamente en Console estas operaciones.

## A1. Anatomía de una solicitud

```http
GET /mi_indice/_search
{
  "query": {
    "match": {
      "descripcion": "mantenimiento aeronaves"
    }
  }
}
```

- `GET` es el método.
- `/mi_indice/_search` es el recurso y la operación.
- El JSON es el cuerpo de la solicitud.
- La respuesta traerá `hits.total`, `hits.hits`, `_id`, `_score` y `_source`.

## A2. Console · crear índice pequeño

Copia esto en Console cuando tengas acceso a Elastic.

```http
PUT s07-demo
{
  "mappings": {
    "properties": {
      "titulo": {"type": "text", "analyzer": "spanish"},
      "categoria": {"type": "keyword"}
    }
  }
}
```

**Cómo se lee.** `titulo` se analiza porque buscaremos palabras dentro. `categoria` queda exacta porque sirve para filtros.

**Evidencia esperada.** Respuesta con `acknowledged: true`.

## A3. Console · observar analyzer

```http
POST s07-demo/_analyze
{
  "analyzer": "spanish",
  "text": "Servicios de mantenimiento de las aeronaves"
}
```

**Qué debes mirar.** La lista `tokens`. No esperes que las palabras salgan exactamente iguales: el analyzer puede normalizar, eliminar palabras frecuentes y aplicar stemming.

**Error frecuente.** Creer que analyzer es búsqueda semántica. Aquí seguimos en búsqueda léxica.

## A4. Console · indexar tres documentos y buscar

```http
PUT s07-demo/_doc/D1
{"titulo": "Mantenimiento de aeronaves KFIR", "categoria": "Defensa"}

PUT s07-demo/_doc/D2
{"titulo": "Construcción de centro aeronáutico", "categoria": "Infraestructura"}

GET s07-demo/_search
{
  "query": {
    "match": {"titulo": "mantenimiento aeronaves"}
  }
}
```

**Para llevar.** Python hará esto mismo, pero de forma repetible y sobre muchos documentos.

# B · Preparar Python

Ahora sí automatizamos. Este bloque no enseña Python desde cero; usa Python para operar Elasticsearch.

In [ ]:
!pip -q install -U elasticsearch pandas tabulate

## B1. Importar librerías y cargar corpus

**Qué queremos hacer.** Cargar el corpus contractual de S6 y convertirlo en documentos únicos por proceso.

**Qué debes poder explicar.** `pandas` prepara los datos; Elasticsearch todavía no está involucrado.

In [ ]:
import pandas as pd
import numpy as np
import json, re, math, unicodedata
from collections import Counter
from pathlib import Path
from IPython.display import display

URL_S06 = 'https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s06_contexto_relacional.csv'

bruto = pd.read_csv(URL_S06, dtype=str, keep_default_na=False)
columnas = ['tipo_registro','entidad','departamento_entidad','id_proceso','nombre_proceso','descripcion','modalidad','url_secop']
for col in columnas:
    if col not in bruto.columns:
        bruto[col] = ''

corpus = bruto[columnas].drop_duplicates('id_proceso').reset_index(drop=True).copy()
corpus['texto_busqueda'] = (corpus['nombre_proceso'] + ' ' + corpus['descripcion']).str.strip()

print('Filas fuente:', len(bruto))
print('Procesos únicos:', len(corpus))
display(corpus.head(3))

### Cómo se lee

Si estás usando el corpus principal, debes observar **2.109 filas fuente** y **1.994 procesos únicos**.

**Qué NO concluye.** No significa que el corpus sea todo SECOP; es el corpus trabajado en la ruta del curso.

**Si falla.** Revisa conexión a internet o el enlace del archivo versionado.

# C · Línea base y BM25 didáctico

Antes del motor real, comparamos una búsqueda literal con un ranking local. Esto sirve para entender por qué `contains()` no basta.

In [ ]:
consulta_demo = 'mantenimiento aeronaves'
terminos = consulta_demo.lower().split()

mask = pd.Series(True, index=corpus.index)
for termino in terminos:
    mask &= corpus['texto_busqueda'].str.lower().str.contains(termino, regex=False, na=False)

literal = corpus.loc[mask, ['id_proceso','entidad','nombre_proceso','url_secop']]
print('Coincidencias literales:', len(literal))
display(literal.head(10))

## C1. Qué hace este código

- `terminos` separa la consulta en palabras.
- `mask` empieza como `True` para todas las filas.
- Cada término restringe la máscara con `contains`.

**Límite.** Todas las coincidencias quedan empatadas; todavía no hay relevancia.

In [ ]:
STOP = {'de','la','el','los','las','del','y','en','para','por','un','una','con','al','se','su','sus','que','es'}

def tokens_didacticos(texto):
    texto = unicodedata.normalize('NFKD', str(texto))
    texto = ''.join(c for c in texto if not unicodedata.combining(c)).lower()
    tokens = re.findall(r'[a-z0-9]+', texto)
    return [t for t in tokens if len(t)>2 and t not in STOP]

def _bm25_scores(textos, consulta, k1=1.2, b=0.75):
    docs = [tokens_didacticos(x) for x in textos]
    N = len(docs)
    lens = np.array([len(d) for d in docs], dtype=float)
    avgdl = lens.mean() if N else 0
    df = Counter()
    for d in docs:
        df.update(set(d))
    q = tokens_didacticos(consulta)
    scores = []
    for i,d in enumerate(docs):
        tf = Counter(d); score = 0.0
        for t in q:
            f = tf.get(t,0)
            if not f: continue
            n = df.get(t,0)
            idf = math.log(1+(N-n+0.5)/(n+0.5))
            den = f + k1*(1-b+b*(lens[i]/avgdl if avgdl else 0))
            score += idf*(f*(k1+1))/den
        scores.append(score)
    return np.array(scores)

def buscar_bm25_local(df, consulta, peso_nombre=1.0):
    scores = peso_nombre*_bm25_scores(df['nombre_proceso'], consulta) + _bm25_scores(df['descripcion'], consulta)
    out = df.copy(); out['score'] = scores
    out = out[out['score']>0].sort_values(['score','id_proceso'], ascending=[False, True]).reset_index(drop=True)
    out.insert(0,'rank',range(1,len(out)+1))
    return out

In [ ]:
ranking_local = buscar_bm25_local(corpus, consulta_demo)
display(ranking_local[['rank','id_proceso','entidad','nombre_proceso','score','url_secop']].head(10))

## C2. Cómo interpretar BM25 local

El score local combina frecuencia, rareza y longitud. Es útil para entender ranking.

**No compares** este número con el `_score` de Elasticsearch. El analyzer, la implementación y la configuración son diferentes.

# D · Conectar Python con Elasticsearch

La conexión ocurre después de entender la API en Console. Ahora Python enviará solicitudes al servicio.

In [ ]:
from getpass import getpass
from elasticsearch import Elasticsearch

USAR_ELASTIC = True #@param {type:'boolean'}
client = None
conexion_error = ''

if USAR_ELASTIC:
    endpoint = input('Project URL de Elasticsearch: ').strip()
    api_key = getpass('API key del proyecto: ').strip()
    try:
        client = Elasticsearch(endpoint, api_key=api_key, request_timeout=30)
        info = client.info()
        print('Conexión verificada')
        print(info.get('tagline','Elasticsearch'))
    except Exception as e:
        conexion_error = f'{type(e).__name__}: {e}'
        print('No se pudo conectar:', conexion_error)
else:
    print('Modo local elegido')

## D1. Línea por línea

- `endpoint`: a qué servicio llamo.
- `api_key`: con qué autorización.
- `client`: objeto Python que sabe enviar solicitudes a Elasticsearch.
- `client.info()`: prueba mínima. Si esto falla, no tiene sentido crear índices o queries.

**Errores típicos.** 401/403 = clave; timeout = URL o red.

# E · Mapping, índice y analyzer real

El mapping es la decisión central: qué campo se busca como texto y qué campo se conserva exacto para filtros.

In [ ]:
ALIAS = 'equipo_demo' #@param {type:'string'}

def slug(valor):
    valor = unicodedata.normalize('NFKD', str(valor))
    valor = ''.join(c for c in valor if not unicodedata.combining(c)).lower()
    return re.sub(r'[^a-z0-9]+','-',valor).strip('-')[:35] or 'equipo-demo'

INDEX_NAME = f's07-compras-claras-{slug(ALIAS)}'
print(INDEX_NAME)

mappings = {
    'properties': {
        'id_proceso': {'type':'keyword'},
        'tipo_registro': {'type':'keyword'},
        'entidad': {'type':'keyword'},
        'departamento_entidad': {'type':'keyword'},
        'modalidad': {'type':'keyword'},
        'nombre_proceso': {'type':'text','analyzer':'spanish'},
        'descripcion': {'type':'text','analyzer':'spanish'},
        'url_secop': {'type':'keyword','index': False}
    }
}
pd.DataFrame([{'campo':k, **v} for k,v in mappings['properties'].items()])

## E1. Código explicado

- `INDEX_NAME` evita que todos escriban sobre el mismo índice.
- `keyword` conserva valores completos para filtros.
- `text` activa analyzer y búsqueda de texto completo.
- `url_secop` se guarda pero no se indexa porque no necesitamos buscar por la URL.

In [ ]:
if client is not None:
    if client.indices.exists(index=INDEX_NAME):
        client.indices.delete(index=INDEX_NAME)
        print('Índice anterior eliminado')
    resp = client.indices.create(index=INDEX_NAME, mappings=mappings)
    print('Índice creado:', resp.get('acknowledged', True))
else:
    print('Sin Elastic: omito creación de índice')

In [ ]:
texto_analisis = 'Servicios de MANTENIMIENTO de las Aeronaves'
if client is not None:
    analisis = client.indices.analyze(index=INDEX_NAME, analyzer='spanish', text=texto_analisis)
    tokens_elastic = [t['token'] for t in analisis['tokens']]
    print(tokens_elastic)
else:
    tokens_elastic = tokens_didacticos(texto_analisis)
    print(tokens_elastic)

## E2. Qué aprendiste con `_analyze`

`client.indices.analyze(...)` es el equivalente Python de `POST índice/_analyze`.

Si los tokens no se parecen a la frase original, eso no es error: es el analyzer trabajando. Este paso te enseña cómo Elasticsearch ve tu texto antes de buscar.

# F · Ingesta con bulk y verificación

Primero entendiste tres documentos en Console. Ahora cargamos 1.994 documentos con Python.

In [ ]:
from elasticsearch.helpers import bulk

campos = ['id_proceso','tipo_registro','entidad','departamento_entidad','modalidad','nombre_proceso','descripcion','url_secop']
documentos = corpus[campos].fillna('').astype(str).to_dict('records')
print('Documentos preparados:', len(documentos))

if client is not None:
    acciones = ({'_index': INDEX_NAME, '_id': d['id_proceso'], '_source': d} for d in documentos)
    ok, errores = bulk(client, acciones, refresh=True, raise_on_error=False)
    print('Documentos indexados:', ok)
    print('Errores bulk:', len(errores))
else:
    ok, errores = 0, []
    print('Sin Elastic: ingesta omitida')

## F1. Anatomía de una acción bulk

- `_index`: índice de destino.
- `_id`: identificador estable del documento.
- `_source`: documento JSON original.

**Evidencia esperada.** Con Elastic real y corpus completo: `Documentos indexados: 1994` y `Errores bulk: 0`.

In [ ]:
if client is not None:
    conteo_elastic = client.count(index=INDEX_NAME)['count']
    print('Conteo local:', len(corpus))
    print('Conteo Elasticsearch:', conteo_elastic)
    assert conteo_elastic == len(corpus)
else:
    conteo_elastic = None
    print('Conteo local:', len(corpus))

# G · Query ladder

Construimos una consulta por capas. Cada capa añade una decisión y debe poder explicarse.

In [ ]:
def mostrar_hits(resp):
    filas=[]
    for pos,h in enumerate(resp.get('hits',{}).get('hits',[]), start=1):
        s=h.get('_source',{})
        frag=[]
        for piezas in h.get('highlight',{}).values():
            frag += piezas
        filas.append({'rank':pos,'id_proceso':s.get('id_proceso',''),'score':h.get('_score'), 'entidad':s.get('entidad',''), 'nombre_proceso':s.get('nombre_proceso',''), 'highlight':' ... '.join(frag), 'url_secop':s.get('url_secop','')})
    return pd.DataFrame(filas)

def buscar_elastic(campos, consulta, con_highlight=False):
    q={'bool':{'must':[{'multi_match':{'query':consulta,'fields':campos}}], 'filter':[{'term':{'tipo_registro':'historico_adjudicado'}}]}}
    kwargs={'index':INDEX_NAME, 'query':q, 'size':5}
    if con_highlight:
        kwargs['highlight']={'fields':{'nombre_proceso':{},'descripcion':{}}}
    return client.search(**kwargs)

In [ ]:
if client is not None:
    resp_match = client.search(index=INDEX_NAME, query={'match': {'descripcion': consulta_demo}}, size=5)
    tabla_match = mostrar_hits(resp_match)
else:
    tabla_match = ranking_local.head(5)[['rank','id_proceso','score','entidad','nombre_proceso','url_secop']]
display(tabla_match)

## G1. `match` explicado

`match` busca texto en un solo campo. La consulta se analiza con el analyzer del campo.

**Qué debes leer en la respuesta.** `_score` ordena; `_source` trae el documento; `hits.total` dice cuántos resultados hubo.

In [ ]:
if client is not None:
    tabla_A = mostrar_hits(buscar_elastic(['nombre_proceso','descripcion'], consulta_demo, con_highlight=True))
else:
    tabla_A = buscar_bm25_local(corpus, consulta_demo, peso_nombre=1).head(5)
    tabla_A['highlight']=''
display(tabla_A[['rank','id_proceso','score','nombre_proceso','highlight','url_secop']])

## G2. `multi_match`, `filter` y `highlight`

Esta configuración busca en `nombre_proceso` y `descripcion`, filtra por `tipo_registro` y muestra fragmentos coincidentes.

- La parte textual aporta score.
- El filtro restringe sin puntuar.
- El highlight ayuda a revisar por qué apareció el documento.

In [ ]:
if client is not None:
    tabla_B = mostrar_hits(buscar_elastic(['nombre_proceso^3','descripcion'], consulta_demo, con_highlight=True))
else:
    tabla_B = buscar_bm25_local(corpus, consulta_demo, peso_nombre=3).head(5)
    tabla_B['highlight']=''
display(tabla_B[['rank','id_proceso','score','nombre_proceso','highlight','url_secop']])

## G3. Boost explicado

`nombre_proceso^3` significa: una coincidencia en el nombre debe pesar más que una coincidencia en la descripción.

No significa que B sea mejor. Solo significa que cambiaste una decisión de ranking. Ahora debes medir si ayudó.

# H · Evaluar relevancia con Precision@5

Antes de etiquetar, define relevancia. No ajustes la definición para favorecer tu ranking.

In [ ]:
CRITERIO_RELEVANCIA = 'Un resultado es relevante si el proceso trata directamente la necesidad textual de la consulta.' #@param {type:'string'}

print('Criterio:', CRITERIO_RELEVANCIA)

In [ ]:
def etiquetar_top5(tabla, nombre):
    etiquetas=[]
    print(f'\nCONFIGURACIÓN {nombre}')
    for _,fila in tabla.head(5).iterrows():
        print('\nRank:', int(fila['rank']))
        print('ID:', fila['id_proceso'])
        print('Nombre:', fila['nombre_proceso'])
        valor = input('¿Relevante? [1=sí, 0=no]: ').strip()
        etiquetas.append(1 if valor=='1' else 0)
    return etiquetas

etiquetas_A = etiquetar_top5(tabla_A, 'A')
etiquetas_B = etiquetar_top5(tabla_B, 'B')
p5_A = sum(etiquetas_A)/5
p5_B = sum(etiquetas_B)/5
print('Precision@5 A:', p5_A)
print('Precision@5 B:', p5_B)
print('Cambio:', round(p5_B-p5_A,3))

## H1. Interpretación

Si B sube documentos pero baja P@5, la modificación no ayudó para este criterio.

Si P@5 sube, aún no tienes una evaluación definitiva: tienes una señal con una consulta y cinco juicios. Para producción se necesitan muchas consultas y ratings.

# I · Reto de transferencia: Sala de redacción

El reto usa noticias de sesiones anteriores. La pregunta cambia: ahora eres editor y quieres recuperar antecedentes útiles.

In [ ]:
URL_NOTICIAS = 'https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/noticias_contratacion_2026.json'
noticias_raw = pd.read_json(URL_NOTICIAS)
noticias = noticias_raw.copy().reset_index(drop=True)
noticias['id_noticia'] = noticias.index.map(lambda i: f'N{i+1:04d}')
for col in ['titulo','subtitulo','cuerpo','categoria','seccion','url']:
    if col not in noticias.columns:
        noticias[col]=''
if 'premium' not in noticias.columns:
    noticias['premium']=False
if 'publicado' not in noticias.columns:
    noticias['publicado']='2026-01-01'
if 'etiquetas' not in noticias.columns:
    noticias['etiquetas']=''
noticias['etiquetas_texto'] = noticias['etiquetas'].astype(str)
print('Noticias:', len(noticias))
display(noticias[['id_noticia','titulo','categoria','premium']].head())

## I1. Diseña el mapping de noticias

Piensa antes de ejecutar:

- `titulo`, `subtitulo`, `cuerpo`: normalmente `text`.
- `categoria`, `seccion`: normalmente `keyword`.
- `premium`: `boolean`.
- `publicado`: `date`.
- `etiquetas`: debes justificar si las tratas como texto, keyword o ambas.

In [ ]:
INDEX_NEWS = f's07-noticias-{slug(ALIAS)}'

mappings_news = {'properties': {
    'id_noticia': {'type':'keyword'},
    'titulo': {'type':'text','analyzer':'spanish'},
    'subtitulo': {'type':'text','analyzer':'spanish'},
    'cuerpo': {'type':'text','analyzer':'spanish'},
    'etiquetas_texto': {'type':'text','analyzer':'spanish'},
    'categoria': {'type':'keyword'},
    'seccion': {'type':'keyword'},
    'premium': {'type':'boolean'},
    'publicado': {'type':'date'},
    'url': {'type':'keyword','index':False}
}}

if client is not None:
    if client.indices.exists(index=INDEX_NEWS):
        client.indices.delete(index=INDEX_NEWS)
    client.indices.create(index=INDEX_NEWS, mappings=mappings_news)
    docs = noticias[['id_noticia','titulo','subtitulo','cuerpo','etiquetas_texto','categoria','seccion','premium','publicado','url']].to_dict('records')
    ok_news, err_news = bulk(client, ({'_index':INDEX_NEWS,'_id':d['id_noticia'],'_source':d} for d in docs), refresh=True, raise_on_error=False)
    print('Noticias indexadas:', ok_news, 'errores:', len(err_news))
else:
    print('Sin Elastic: reto queda como diseño y reflexión')

## I2. Brief editorial

Tu alias asigna un tema. Debes comparar A y B manteniendo `premium = false`.

In [ ]:
BRIEFS = [
    ('control fiscal','antecedentes sobre control fiscal y contratación'),
    ('sobrecostos','antecedentes sobre sobrecostos y obras públicas'),
    ('licitaciones','historias sobre licitaciones y adjudicaciones'),
    ('corrupción contratación','indicios o denuncias sobre corrupción en contratación'),
    ('inteligencia artificial','noticias sobre IA y sector público')
]
brief = BRIEFS[sum(ord(c) for c in slug(ALIAS)) % len(BRIEFS)]
print('Consulta editorial:', brief[0])
print('Criterio:', brief[1])

In [ ]:
def buscar_news(campos, consulta):
    q={'bool': {'must':[{'multi_match': {'query': consulta, 'fields': campos}}], 'filter':[{'term': {'premium': False}}]}}
    return client.search(index=INDEX_NEWS, query=q, highlight={'fields':{'titulo':{},'subtitulo':{},'cuerpo':{}}}, size=5)

if client is not None:
    news_A = mostrar_hits(buscar_news(['titulo','subtitulo','cuerpo'], brief[0]))
    news_B = mostrar_hits(buscar_news(['titulo^4','etiquetas_texto^3','subtitulo^2','cuerpo'], brief[0]))
    print('A')
    display(news_A)
    print('B')
    display(news_B)
else:
    news_A = pd.DataFrame(); news_B = pd.DataFrame()

## I3. Cierre del reto

Decide si A o B sirve mejor para el editor. Usa P@5, un resultado concreto, un falso positivo y una mejora siguiente.

Este reto puede iniciarse en clase y cerrarse como entrega corta si el tiempo no alcanza.

# J · Exportación final

In [ ]:
from datetime import datetime, timezone

DECISION = 'Escribe qué ranking defenderías y por qué.' #@param {type:'string'}
LIMITE = 'Escribe qué no demuestra este ranking.' #@param {type:'string'}

config = {
    'fecha_utc': datetime.now(timezone.utc).isoformat(),
    'alias': slug(ALIAS),
    'indice_contratos': INDEX_NAME,
    'indice_noticias': INDEX_NEWS,
    'consulta_demo': consulta_demo,
    'criterio_relevancia': CRITERIO_RELEVANCIA,
    'precision_at_5_A': p5_A if 'p5_A' in globals() else None,
    'precision_at_5_B': p5_B if 'p5_B' in globals() else None,
    'decision': DECISION,
    'limite': LIMITE,
    'conexion_error': conexion_error
}
Path('s07_config_busqueda.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')

tabla_A.assign(configuracion='A').to_csv('s07_resultados_busqueda.csv', index=False, encoding='utf-8-sig')
Path('hito_s07_relevancia.md').write_text(f'# Hito S07\n\nDecisión: {DECISION}\n\nLímite: {LIMITE}\n', encoding='utf-8')
print('Archivos creados: s07_config_busqueda.json, s07_resultados_busqueda.csv, hito_s07_relevancia.md')

## Pasaporte de habilidades

Marca mentalmente si ya puedes hacerlo sin ayuda:

- entrar a Console;
- crear índice;
- diseñar mapping;
- usar `_analyze`;
- indexar documentos;
- leer `_search`;
- usar `match`, `multi_match`, boost, `filter` y `highlight`;
- conectar Python;
- cargar con `bulk`;
- evaluar con Precision@5;
- repetir en noticias.

Si algo no puedes explicarlo, no lo des por aprendido.